In [ ]:
import os
import importlib
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import training_new18 as training
training = importlib.reload(training)  # Ensure the latest training script is loaded.

save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

# Training configuration for ParamNet.
cfg = training.TrainingConfig(
    train_file='Tra_data1e4_All.npz',
    val_file='Var_data1e4_ALL.npz',
    Test_file='Test_data1e4_ALL.npz',

    batch_size=1024,
    learning_rate=5e-4,
    weight_decay=1e-6,
    epochs=220,
    hidden_dim=512,
    dropout=0.3,
    patience=30,
    save_dir=save_dir,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    window_size=100,
    seq_overlap=0,

    # Physics loss schedule: stronger early, weaker later.
    physics_weight=0.7,
    physics_warmup_epochs=40,
    physics_peak_epoch_ratio=0.35,
    physics_min_ratio=0.15,

    # Pressure auxiliary augmentation settings.
    include_pressure_aux=True,
    pressure_window_jitter_rel=0.3,
    pressure_window_jitter_abs=0.0,
    pressure_jitter_std=0.0,
    pressure_dropout_prob=0.0,
    pressure_shuffle_prob=0.0,
    gamma_p_corr_weight=0.06,

    # Lightweight time-series augmentation.
    ts_noise_std=0.01,
    ts_gain_jitter=0.08,

    num_workers=2,
    grad_clip=1.0,
)

print(f"Device: {cfg.device}")

# Build datasets and dataloaders.
train_dataset = training.BrownianDataset(
    cfg.train_file, window_size=cfg.window_size, overlap=cfg.seq_overlap,
    is_train=True,
    include_pressure_aux=cfg.include_pressure_aux,
    pressure_window_jitter_rel=cfg.pressure_window_jitter_rel,
    pressure_window_jitter_abs=cfg.pressure_window_jitter_abs,
    pressure_jitter_std=cfg.pressure_jitter_std,
    pressure_dropout_prob=cfg.pressure_dropout_prob,
    pressure_shuffle_prob=cfg.pressure_shuffle_prob,
    ts_noise_std=cfg.ts_noise_std,
    ts_gain_jitter=cfg.ts_gain_jitter,
)
val_dataset = training.BrownianDataset(
    cfg.val_file, window_size=cfg.window_size, overlap=cfg.seq_overlap,
    is_train=False,
    include_pressure_aux=cfg.include_pressure_aux,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device == 'cuda'),
    drop_last=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=max(1, cfg.num_workers // 2),
    pin_memory=(cfg.device == 'cuda')
)

print(f"Training windows: {len(train_dataset)}")
if hasattr(train_dataset, "fs_values"):
    print(f"Training sampling-rate range: {train_dataset.fs_values.min():.0f}-{train_dataset.fs_values.max():.0f} Hz")
print(f"Validation windows: {len(val_dataset)}")
if hasattr(val_dataset, "fs_values"):
    print(f"Validation sampling-rate range: {val_dataset.fs_values.min():.0f}-{val_dataset.fs_values.max():.0f} Hz")

print("Start training (ParamNetV2 + pressure perturbation + scheduled physics loss)...")
model = training.train_model(cfg)

In [ ]:
from pathlib import Path
from torch.serialization import add_safe_globals
import importlib
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import savemat
from torch.utils.data import DataLoader
import training_new18 as training

training = importlib.reload(training)
add_safe_globals([np.core.multiarray.scalar])

# Load the best checkpoint and run test inference.
save_dir = './checkpoints'
ckpt_path = Path(save_dir) / 'best_model.pth'
if not ckpt_path.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')

# Keep training and model window sizes consistent.
train_window_size = 100
# Test window size can be changed for experiments.
test_window_size = 100

# Select an available test file.
base = Path.cwd()
test_candidates = [
    base / 'Test_data1e4_ALL.npz',
    base / 'Test_data1e4_ALL.npz',
]

test_file = None
for p in test_candidates:
    if p.is_file():
        test_file = str(p.resolve())
        break
if test_file is None:
    raise FileNotFoundError('Test file not found: Test_data1e4_new.npz or Test_data1e4_ALL.npz')

print(f'Checkpoint: {ckpt_path}')
print(f'Test file: {test_file}')

cfg_base = training.TrainingConfig(
    train_file='Tra_data1e4_All.npz',
    val_file='Var_data1e4_ALL.npz',
    Test_file=test_file,
    batch_size=1024,
    learning_rate=5e-4,
    weight_decay=1e-6,
    epochs=220,
    hidden_dim=512,
    dropout=0.3,
    patience=30,
    save_dir=save_dir,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    window_size=train_window_size,
    seq_overlap=0,
    include_pressure_aux=True,
    num_workers=2,
)

# Inference config: only test file and window-size dependent values differ.
cfg_test = training.TrainingConfig(
    train_file=cfg_base.train_file,
    val_file=cfg_base.val_file,
    Test_file=cfg_base.Test_file,
    batch_size=cfg_base.batch_size,
    learning_rate=cfg_base.learning_rate,
    weight_decay=cfg_base.weight_decay,
    epochs=cfg_base.epochs,
    hidden_dim=cfg_base.hidden_dim,
    dropout=cfg_base.dropout,
    patience=cfg_base.patience,
    save_dir=cfg_base.save_dir,
    device=cfg_base.device,
    window_size=test_window_size,
    seq_overlap=0,
    include_pressure_aux=cfg_base.include_pressure_aux,
    num_workers=cfg_base.num_workers,
)

print(f'Training window size: {cfg_base.window_size}')
print(f'Test window size: {cfg_test.window_size}')

model = training.ParamNetV2(
    window_size=cfg_base.window_size,
    aux_dim=6,
    hid=cfg_base.hidden_dim,
    dropout=cfg_base.dropout,
).to(cfg_base.device)

checkpoint = torch.load(ckpt_path, map_location=cfg_base.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_dataset = training.BrownianDataset(
    cfg_test.Test_file,
    window_size=cfg_test.window_size,
    overlap=0,
    is_train=False,
    include_pressure_aux=cfg_test.include_pressure_aux,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg_test.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(cfg_test.device == 'cuda'),
)

# Measure end-to-end inference latency and throughput.
if cfg_test.device == 'cuda':
    torch.cuda.synchronize()
t0 = time.perf_counter()
predictions = training.predict(model, test_loader, cfg_test.device)
if cfg_test.device == 'cuda':
    torch.cuda.synchronize()
t1 = time.perf_counter()

infer_total_s = t1 - t0
num_windows = len(test_dataset)
infer_per_window_ms = infer_total_s / max(num_windows, 1) * 1000.0
throughput = num_windows / max(infer_total_s, 1e-12)

print(f'Total inference time: {infer_total_s:.4f} s')
print(f'Total windows: {num_windows}')
print(f'Average inference latency: {infer_per_window_ms:.4f} ms/window')
print(f'Inference throughput: {throughput:.2f} windows/s')

k_rel_error = np.mean(np.abs(predictions['k_pred'] - predictions['k_true']) / predictions['k_true'])
g_rel_error = np.mean(np.abs(predictions['g_pred'] - predictions['g_true']) / predictions['g_true'])
print('Mean relative error on test set:')
print(f'k error: {k_rel_error * 100:.2f}%')
print(f'g error: {g_rel_error * 100:.2f}%')

k_errors = np.abs(predictions['k_pred'] - predictions['k_true']) / predictions['k_true'] * 100
g_errors = np.abs(predictions['g_pred'] - predictions['g_true']) / predictions['g_true'] * 100

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.hist(k_errors, bins=50, alpha=0.7, label=f'Mean: {np.mean(k_errors):.2f}%')
plt.title('k Relative Error Distribution (%)')
plt.xlabel('Relative error (%)')
plt.ylabel('Number of samples')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.hist(g_errors, bins=50, alpha=0.7, label=f'Mean: {np.mean(g_errors):.2f}%')
plt.title('g Relative Error Distribution (%)')
plt.xlabel('Relative error (%)')
plt.ylabel('Number of samples')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

export_dict = {
    'k_pred': predictions['k_pred'].astype(np.float32).reshape(-1, 1),
    'g_pred': predictions['g_pred'].astype(np.float32).reshape(-1, 1),
    'k_true': predictions['k_true'].astype(np.float32).reshape(-1, 1),
    'g_true': predictions['g_true'].astype(np.float32).reshape(-1, 1),
    'infer_total_s': np.array([[infer_total_s]], dtype=np.float32),
    'infer_per_window_ms': np.array([[infer_per_window_ms]], dtype=np.float32),
    'throughput_windows_per_s': np.array([[throughput]], dtype=np.float32),
}
training.visualize_predictions(predictions)